# Proyecto: Modelado Espaciotemporal y Predicción de Picos de Malaria en Colombia mediante Autómatas Celulares y Machine Learning

## Integrantes

- Daniel Solarte Chaverra

## Entorno de ejecución

### Hardware

`Apple MacBook Pro` con procesador **Apple M1 Pro** (10 núcleos físicos: 8 núcleos de alto rendimiento *Firestorm* + 2 núcleos de alta eficiencia *Icestorm*, sin *hyperthreading*, por lo que núcleos físicos = lógicos = 10) y **16 GB de memoria unificada LPDDR5**.

### Dependencias

- **Python:** `3.14.3`
- **Pandas:** `3.0.2`
- **NumPy:** `2.4.4`
- **Scikit-Learn:** `1.9.1`

In [89]:
from concurrent.futures import ThreadPoolExecutor
from enum import Enum
from IPython.display import HTML
import math
import matplotlib as mpl
import matplotlib.animation as animation
import matplotlib.pyplot as plt
import numpy as np
import os
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import train_test_split
import time
from typing import Any, Optional

mpl.rcParams["animation.embed_limit"] = 100.0
pd.options.plotting.backend = "plotly"

## Utilidades

In [ ]:
class ModelType(Enum):
    COLOMBIA = "Nivel Nacional"
    DEPARTMENT = "Nivel Departamental"
    CITY = "Nivel Municipal"


class MalariaType(Enum):
    VIVAX = "vivax"
    FALCIPARUM = "falciparum"


## Tratamiento y Análisis exploratorio multiescala

## Nacional

In [44]:
df_colombia = pd.read_parquet("./GAM_regional_enriquecido.parquet")
df_colombia.head()

,region,fecha,año,mes,casos_totales,vivax,falciparum,malarie,complicada,mixta,...,tercera_cobertura,tercera_cobertura_pct,uso_suelo_n_pixeles_validos,poblacion_total,poblacion_cabecera,poblacion_rural,porcentaje_rural,IPA_mensual,IPA_vivax,IPA_falciparum
0,Amazonia,2007-01-01,2007,1,155,121,34,0.0,0,0,...,Mosaico de agricultura o pasto,5.651958,162517426,858685.0,478026.0,380659.0,44.33,0.1805,0.1409,0.0396
1,Amazonia,2007-02-01,2007,2,335,239,86,0.0,0,10,...,Mosaico de agricultura o pasto,5.651958,162517426,858685.0,478026.0,380659.0,44.33,0.3901,0.2783,0.1002
2,Amazonia,2007-03-01,2007,3,432,315,114,0.0,1,2,...,Mosaico de agricultura o pasto,5.651958,162517426,858685.0,478026.0,380659.0,44.33,0.5031,0.3668,0.1328
3,Amazonia,2007-04-01,2007,4,646,389,250,0.0,0,7,...,Mosaico de agricultura o pasto,5.651958,162517426,858685.0,478026.0,380659.0,44.33,0.7523,0.4530,0.2911
4,Amazonia,2007-05-01,2007,5,604,393,202,0.0,0,9,...,Mosaico de agricultura o pasto,5.651958,162517426,858685.0,478026.0,380659.0,44.33,0.7034,0.4577,0.2352


### Código de cada región dentro del DataFrame

In [45]:
df_regions = df_colombia["region"].drop_duplicates().reset_index().drop(columns=["index"])

Regions = Enum("Regions", {
    row["region"].upper(): row["region"]
        for _, row in df_regions.iterrows()
})

df_regions

,region
0,Amazonia
1,Andina
2,Caribe
3,Orinoquia
4,Pacifico


### Fechas del DataFrame

| Inicio | Final |
|---|---|
| Enero 2007 | Diciembre 2023 |

In [46]:
df_colombia["fecha"].drop_duplicates()

0     2007-01-01
1     2007-02-01
2     2007-03-01
3     2007-04-01
4     2007-05-01
         ...    
199   2023-08-01
200   2023-09-01
201   2023-10-01
202   2023-11-01
203   2023-12-01
Name: fecha, Length: 204, dtype: datetime64[us]

## Departamental

In [47]:
df_departments = pd.read_parquet("./GAM_departamental_enriquecido.parquet")
df_departments.head()

,codigo_departamento,departamento,region,fecha,año,mes,casos_totales,vivax,falciparum,malarie,...,nbi_total_pct,miseria_total_pct,nbi_vivienda_pct,nbi_servicios_pct,nbi_hacinamiento_pct,nbi_inasistencia_pct,nbi_dependencia_pct,IPA_mensual,IPA_vivax,IPA_falciparum
0,5,Antioquia,Andina,2007-01-01,2007,1,22,14,7,0.0,...,18.459065,4.718356,5.681791,3.924864,4.838187,2.038261,8.349581,0.0039,0.0025,0.0013
1,5,Antioquia,Andina,2007-02-01,2007,2,33,27,6,0.0,...,18.459065,4.718356,5.681791,3.924864,4.838187,2.038261,8.349581,0.0059,0.0048,0.0011
2,5,Antioquia,Andina,2007-03-01,2007,3,37,27,10,0.0,...,18.459065,4.718356,5.681791,3.924864,4.838187,2.038261,8.349581,0.0066,0.0048,0.0018
3,5,Antioquia,Andina,2007-04-01,2007,4,54,41,9,0.0,...,18.459065,4.718356,5.681791,3.924864,4.838187,2.038261,8.349581,0.0097,0.0073,0.0016
4,5,Antioquia,Andina,2007-05-01,2007,5,57,42,10,0.0,...,18.459065,4.718356,5.681791,3.924864,4.838187,2.038261,8.349581,0.0102,0.0075,0.0018


### Código de cada departamento dentro del DataFrame

In [48]:
df_departments_codes = df_departments[["codigo_departamento", "departamento"]] \
                        .drop_duplicates().reset_index().drop(columns=["index"])

Departments = Enum("Departments", {
    department["departamento"].upper(): department["codigo_departamento"]
        for _, department in df_departments_codes.iterrows()
})

df_departments_codes

,codigo_departamento,departamento
0,5,Antioquia
1,8,Atlántico
2,11,Bogotá D.C.
3,13,Bolívar
4,15,Boyacá
5,17,Caldas
6,18,Caquetá
7,19,Cauca
8,20,Cesar
9,23,Córdoba


## Municipal

In [49]:
df_municipal = pd.read_parquet("./GAM_municipal_enriquecido.parquet")
df_municipal.head()

,codigo_municipio,municipio,codigo_departamento,fecha,año,mes,casos_totales,vivax,falciparum,malarie,...,nbi_total_pct,miseria_total_pct,nbi_vivienda_pct,nbi_servicios_pct,nbi_hacinamiento_pct,nbi_inasistencia_pct,nbi_dependencia_pct,IPA_mensual,IPA_vivax,IPA_falciparum
0,5001,MEDELLÍN,5,2007-01-01,2007,1,0,0,0,0.0,...,5.230399,0.417281,0.284643,0.188702,1.555428,1.614882,2.0503,0.0000,0.0000,0.0000
1,5001,MEDELLÍN,5,2007-02-01,2007,2,0,0,0,0.0,...,5.230399,0.417281,0.284643,0.188702,1.555428,1.614882,2.0503,0.0000,0.0000,0.0000
2,5001,MEDELLÍN,5,2007-03-01,2007,3,0,0,0,0.0,...,5.230399,0.417281,0.284643,0.188702,1.555428,1.614882,2.0503,0.0000,0.0000,0.0000
3,5001,MEDELLÍN,5,2007-04-01,2007,4,1,1,0,0.0,...,5.230399,0.417281,0.284643,0.188702,1.555428,1.614882,2.0503,0.0005,0.0005,0.0000
4,5001,MEDELLÍN,5,2007-05-01,2007,5,1,0,1,0.0,...,5.230399,0.417281,0.284643,0.188702,1.555428,1.614882,2.0503,0.0005,0.0000,0.0005


### Código de cada municipio dentro del DataFrame

In [50]:
df_municipal_codes = df_municipal[["codigo_municipio", "municipio"]] \
                        .drop_duplicates().reset_index().drop(columns=["index"])

Cities = Enum("Cities", {
    row["municipio"].upper(): row["codigo_municipio"]
        for _, row in df_municipal_codes.iterrows()
})

df_municipal_codes

,codigo_municipio,municipio
0,5001,MEDELLÍN
1,5002,ABEJORRAL
2,5004,ABRIAQUÍ
3,5021,ALEJANDRÍA
4,5030,AMAGÁ
...,...,...
1115,97889,YAVARATÉ
1116,99001,PUERTO CARREÑO
1117,99524,LA PRIMAVERA
1118,99624,SANTA ROSALÍA


## Modelos de Regresión y Detección de Picos

### Variables

In [54]:
RM_SELECTED_MODEL_TYPE: ModelType = ModelType.COLOMBIA
RM_SELECTED_MALARIA_TYPE: MalariaType | list[MalariaType] = [MalariaType.VIVAX, MalariaType.FALCIPARUM]

#### Utilidades para las variables

In [55]:
# Generate graphics filters title

rm_filters_title = f"{RM_SELECTED_MODEL_TYPE.value} - Todas las regiones"

if RM_SELECTED_MODEL_TYPE in [ModelType.DEPARTMENT, ModelType.CITY]:
    rm_filters_title += " - Todos los departamentos"

if RM_SELECTED_MODEL_TYPE == ModelType.CITY:
    rm_filters_title += " - Todos los municipios"


# Get filtered DataFrame

df_rm_sub = {
    ModelType.COLOMBIA: df_colombia.copy(),
    ModelType.DEPARTMENT: df_departments.copy(),
    ModelType.CITY: df_municipal.copy(),
}[RM_SELECTED_MODEL_TYPE]


print(f"Título: {rm_filters_title}")
print(f"Tamaño del DataFrame: {len(df_rm_sub)} datos")

Título: Nivel Nacional - Todas las regiones
Tamaño del DataFrame: 1020 datos


### Obtención del DataFrame

In [56]:
df_rm_sub = df_rm_sub.copy()

if RM_SELECTED_MODEL_TYPE == ModelType.COLOMBIA:
    df_rm_sub_group = df_rm_sub.groupby(["region"])
elif RM_SELECTED_MODEL_TYPE == ModelType.DEPARTMENT:
    df_rm_sub_group = df_rm_sub.groupby(["codigo_departamento"])
elif RM_SELECTED_MODEL_TYPE == ModelType.CITY:
    df_rm_sub_group = df_rm_sub.groupby(["codigo_municipio"])

# Clima y geografía

df_rm_sub["temperatura_media_Celsius_lag1"] = df_rm_sub_group["temperatura_media_Celsius"].shift(1)
df_rm_sub["temperatura_media_Celsius_lag2"] = df_rm_sub_group["temperatura_media_Celsius"].shift(2)

df_rm_sub["temperatura_min_Celsius_lag1"] = df_rm_sub_group["temperatura_min_Celsius"].shift(1)

df_rm_sub["precipitacion_media_mm_lag1"] = df_rm_sub_group["precipitacion_media_mm"].shift(1)
df_rm_sub["precipitacion_media_mm_lag2"] = df_rm_sub_group["precipitacion_media_mm"].shift(2)
df_rm_sub["precipitacion_media_mm_lag3"] = df_rm_sub_group["precipitacion_media_mm"].shift(3)

df_rm_sub["dias_lluvia_promedio_lag1"] = df_rm_sub_group["dias_lluvia_promedio"].shift(1)

df_rm_sub["evapotranspiracion_acumulada_media_mm_lag1"] = df_rm_sub_group["evapotranspiracion_acumulada_media_mm"].shift(1)

df_rm_sub["humedad_mediana_pct_lag1"] = df_rm_sub_group["humedad_mediana_pct"].shift(1)

# ENSO 3 a 6 meses previos

df_rm_sub["MEI_v2_lag3"] = df_rm_sub["MEI_v2"].shift(3)
df_rm_sub['MEI_v2_lag6'] = df_rm_sub['MEI_v2'].shift(6)

# Otros

df_rm_sub['mes_sin'] = np.sin(2 * np.pi * df_rm_sub['mes'] / 12)
df_rm_sub['mes_cos'] = np.cos(2 * np.pi * df_rm_sub['mes'] / 12)


df_rm_sub = df_rm_sub.dropna()
if RM_SELECTED_MODEL_TYPE == ModelType.COLOMBIA:
    df_rm_sub = df_rm_sub.sort_values(["fecha", "region"])
elif RM_SELECTED_MODEL_TYPE == ModelType.DEPARTMENT:
    df_rm_sub = df_rm_sub.sort_values(["fecha", "codigo_departamento"])
elif RM_SELECTED_MODEL_TYPE == ModelType.CITY:
    df_rm_sub = df_rm_sub.sort_values(["fecha", "codigo_municipio"])

df_rm_sub.head()

,region,fecha,año,mes,casos_totales,vivax,falciparum,malarie,complicada,mixta,...,precipitacion_media_mm_lag1,precipitacion_media_mm_lag2,precipitacion_media_mm_lag3,dias_lluvia_promedio_lag1,evapotranspiracion_acumulada_media_mm_lag1,humedad_mediana_pct_lag1,MEI_v2_lag3,MEI_v2_lag6,mes_sin,mes_cos
207,Andina,2007-04-01,2007,4,208,169,23,1.0,5,10,...,216.710377,193.209358,314.135035,18.233818,116.402806,81.396732,0.64,0.48,0.866025,-0.500000
411,Caribe,2007-04-01,2007,4,743,484,230,0.0,9,20,...,105.139792,65.444234,33.682212,11.597613,164.597619,75.316376,0.64,0.48,0.866025,-0.500000
615,Orinoquia,2007-04-01,2007,4,222,150,61,0.0,1,10,...,163.367788,35.517449,33.616105,13.388664,166.609848,75.310112,0.64,0.48,0.866025,-0.500000
819,Pacifico,2007-04-01,2007,4,612,122,481,2.0,3,4,...,379.406541,353.847216,571.880874,21.601185,96.784579,87.046635,0.64,0.48,0.866025,-0.500000
208,Andina,2007-05-01,2007,5,244,197,20,1.0,7,19,...,322.715159,216.710377,193.209358,21.933266,114.659576,83.723618,0.40,0.91,0.500000,-0.866025


#### Columnas del DataFrame

In [52]:
for column in df_rm_sub.columns:
    print(column)

region
fecha
año
mes
casos_totales
vivax
falciparum
malarie
complicada
mixta
n_tipos_disponibles
MEI_v2
precipitacion_media_mm
precipitacion_sd_mm
precipitacion_mediana_mm
precipitacion_min_mm
precipitacion_max_mm
precipitacion_p25_mm
precipitacion_p75_mm
precipitacion_n_pixeles_validos
temperatura_media_Celsius
temperatura_sd_Celsius
temperatura_mediana_Celsius
temperatura_min_Celsius
temperatura_max_Celsius
temperatura_p25_Celsius
temperatura_p75_Celsius
temperatura_n_pixeles_validos
humedad_media_pct
humedad_sd_pct
humedad_mediana_pct
humedad_min_pct
humedad_max_pct
humedad_p25_pct
humedad_p75_pct
humedad_n_pixeles_validos
evapotranspiracion_acumulada_media_mm
evapotranspiracion_acumulada_sd_mm
evapotranspiracion_acumulada_mediana_mm
evapotranspiracion_acumulada_min_mm
evapotranspiracion_acumulada_max_mm
evapotranspiracion_acumulada_p25_mm
evapotranspiracion_acumulada_p75_mm
evapotranspiracion_n_pixeles_validos
dias_lluvia_promedio
dias_lluvia_sd
dias_lluvia_mediana
dias_lluvia_min


In [53]:
df_rm_sub[["año", "casos_totales", "IPA_mensual", "vivax", "IPA_vivax", "falciparum", "IPA_falciparum"]].groupby(["año"]).sum()

,casos_totales,IPA_mensual,vivax,IPA_vivax,falciparum,IPA_falciparum
año,,,,,,
2007,25764,7.0029,16993,4.9292,8038,1.8686
2008,61360,17.0417,44274,12.4696,16182,4.3139
2009,78063,16.0701,56063,12.1033,21063,3.7540
2010,117103,18.7534,82247,13.9374,32663,4.4813
2011,63431,8.8147,46819,6.1262,15249,2.4777
2012,57777,7.7153,42687,5.7939,14135,1.8001
2013,51350,8.4508,32874,6.1711,17322,2.1233
2014,38189,6.2621,18649,3.7077,18705,2.4147
2015,52788,11.5797,24154,7.2667,27082,3.9358


### Columnas del DataFrame para cada cepa

In [57]:
# Falciparum

FALCIPARUM_COLUMNS = [
    "mes_sin",
    "mes_cos",
    # "temperatura_media_Celsius_lag1",
    "temperatura_min_Celsius_lag1",
    "dias_lluvia_promedio_lag1",
    "precipitacion_media_mm_lag2",
    "evapotranspiracion_acumulada_media_mm_lag1",
    # "humedad_mediana_pct_lag1",
    "MEI_v2_lag3",
]

if RM_SELECTED_MODEL_TYPE == ModelType.COLOMBIA:
    FALCIPARUM_COLUMNS.extend([
        "porcentaje_rural",
        "elevacion_media_m",
    ])
elif RM_SELECTED_MODEL_TYPE == ModelType.DEPARTMENT:
    FALCIPARUM_COLUMNS.extend([
        "nbi_total_pct",
    ])


# Vivax

VIVAX_COLUMNS = [
    "mes_sin",
    "mes_cos",
    "temperatura_media_Celsius_lag1",
    "temperatura_min_Celsius_lag1",
    # "temperatura_media_Celsius_lag2",
    "precipitacion_media_mm_lag2",
    # "precipitacion_media_mm_lag3",
    "humedad_mediana_pct_lag1",
    "MEI_v2_lag3",
    # "MEI_v2_lag6",
]

if RM_SELECTED_MODEL_TYPE == ModelType.COLOMBIA:
    VIVAX_COLUMNS.extend([
        "porcentaje_rural",
        "elevacion_media_m",
    ])
elif RM_SELECTED_MODEL_TYPE == ModelType.DEPARTMENT:
    VIVAX_COLUMNS.extend([
        "nbi_total_pct",
    ])

#### Matriz de correlación: IPA_falciparum vs Covariables

In [58]:
px.imshow(
    df_rm_sub[["IPA_falciparum"] + FALCIPARUM_COLUMNS].corr(),
    text_auto=".2f",
    aspect="auto",
    color_continuous_scale='RdBu_r', # Escala: Rojo (negativa) a Azul (positiva)
    title=f"Matriz de Correlación: IPA_falciparum vs Covariables - {rm_filters_title}"
).show()

#### Matriz de correlación: IPA_vivax vs Covariables

In [59]:
px.imshow(
    df_rm_sub[["IPA_vivax"] + VIVAX_COLUMNS].corr(),
    text_auto=".2f",
    aspect="auto",
    color_continuous_scale='RdBu_r', # Escala: Rojo (negativa) a Azul (positiva)
    title=f"Matriz de Correlación: IPA_vivax vs Covariables - {rm_filters_title}"
).show()

### Separación de los datos de entrenamiento, prueba y predicción

In [60]:
if not isinstance(RM_SELECTED_MALARIA_TYPE, list):
    RM_SELECTED_MALARIA_TYPE = [RM_SELECTED_MALARIA_TYPE]

# Falciparum

x_rm_falciparum = None
y_rm_falciparum = None

if MalariaType.FALCIPARUM in RM_SELECTED_MALARIA_TYPE:
    x_rm_falciparum = df_rm_sub[FALCIPARUM_COLUMNS]
    y_rm_falciparum = df_rm_sub["IPA_falciparum"]

if x_rm_falciparum is not None and y_rm_falciparum is not None:
    # Primera división: Separar el 15% final para el set de Predicción "ciega" (Holdout)
    # Esto deja el 85% inicial como datos temporales para entrenar/evaluar
    x_rm_falciparum_temp, x_rm_falciparum_pred_blind, y_rm_falciparum_temp, y_rm_falciparum_actual_blind = train_test_split(
        x_rm_falciparum,
        y_rm_falciparum,
        test_size=0.15,
        shuffle=False,
    )

    # Segunda división: Del bloque temporal (85%), tomamos el 20% final para el Test (evaluación del modelo)
    # y el resto será el set definitivo de Entrenamiento
    x_rm_falciparum_train, x_rm_falciparum_test, y_rm_falciparum_train, y_rm_falciparum_test = train_test_split(
        x_rm_falciparum_temp,
        y_rm_falciparum_temp,
        test_size=0.20,
        shuffle=False,
    )

    print("=== FALCIPARUM ===")
    print(f"Entrenamiento (Pasado remoto): {len(x_rm_falciparum_train)} filas")
    print(f"Test (Pasado reciente): {len(x_rm_falciparum_test)} filas")
    print(f"Predicción (Futuro 'desconocido'): {len(x_rm_falciparum_pred_blind)} filas")
    print("")


# Vivax

x_rm_vivax = None
y_rm_vivax = None

if MalariaType.VIVAX in RM_SELECTED_MALARIA_TYPE:
    x_rm_vivax = df_rm_sub[VIVAX_COLUMNS]
    y_rm_vivax = df_rm_sub["IPA_vivax"]

if x_rm_vivax is not None and y_rm_vivax is not None:
    # Primera división: Separar el 15% final para el set de Predicción "ciega" (Holdout)
    # Esto deja el 85% inicial como datos temporales para entrenar/evaluar
    x_rm_vivax_temp, x_rm_vivax_pred_blind, y_rm_vivax_temp, y_rm_vivax_actual_blind = train_test_split(
        x_rm_vivax,
        y_rm_vivax,
        test_size=0.15,
        shuffle=False,
    )

    # Segunda división: Del bloque temporal (85%), tomamos el 20% final para el Test (evaluación del modelo)
    # y el resto será el set definitivo de Entrenamiento
    x_rm_vivax_train, x_rm_vivax_test, y_rm_vivax_train, y_rm_vivax_test = train_test_split(
        x_rm_vivax_temp,
        y_rm_vivax_temp,
        test_size=0.20,
        shuffle=False,
    )

    print("=== VIVAX ===")
    print(f"Entrenamiento (Pasado remoto): {len(x_rm_vivax_train)} filas")
    print(f"Test (Pasado reciente): {len(x_rm_vivax_test)} filas")
    print(f"Predicción (Futuro 'desconocido'): {len(x_rm_vivax_pred_blind)} filas")


=== FALCIPARUM ===
Entrenamiento (Pasado remoto): 680 filas
Test (Pasado reciente): 171 filas
Predicción (Futuro 'desconocido'): 151 filas

=== VIVAX ===
Entrenamiento (Pasado remoto): 680 filas
Test (Pasado reciente): 171 filas
Predicción (Futuro 'desconocido'): 151 filas


### Aplicación del modelo

In [61]:
# Falciparum

if x_rm_falciparum is not None:
    model_rm_falciparum = GradientBoostingRegressor(
        n_estimators=1_000,      # Cantidad de árboles secuenciales a construir
        learning_rate=0.005,     # Qué tanto peso (corrección) aporta cada árbol
        max_depth=5,           # Profundidad máxima de cada árbol (se recomiendan árboles pequeños o "débiles")
        random_state=42        # Semilla para que los resultados sean reproducibles
    )
    model_rm_falciparum.fit(x_rm_falciparum_train, y_rm_falciparum_train)

    # Make predictions on each set
    y_rm_falciparum_train_pred = model_rm_falciparum.predict(x_rm_falciparum_train)
    y_rm_falciparum_test_pred = model_rm_falciparum.predict(x_rm_falciparum_test)
    y_rm_falciparum_blind_pred = model_rm_falciparum.predict(x_rm_falciparum_pred_blind)

    # Variables
    mae_rm_falciparum = mean_absolute_error(y_rm_falciparum_test, y_rm_falciparum_test_pred)
    mse_rm_falciparum = mean_squared_error(y_rm_falciparum_test, y_rm_falciparum_test_pred)

    print("=== Falciparum ===")
    print("Mean Absolute Error (MAE): {0}".format(mae_rm_falciparum))
    print("Mean Squared Error (MSE): {0}".format(mse_rm_falciparum))
    print("")


# Vivax

if x_rm_vivax is not None:
    model_rm_vivax = GradientBoostingRegressor(
        n_estimators=2_000,      # Cantidad de árboles secuenciales a construir
        learning_rate=0.05,     # Qué tanto peso (corrección) aporta cada árbol
        max_depth=2,           # Profundidad máxima de cada árbol (se recomiendan árboles pequeños o "débiles")
        random_state=42        # Semilla para que los resultados sean reproducibles
    )
    model_rm_vivax.fit(x_rm_vivax_train, y_rm_vivax_train)    

    # Make predictions on each set
    y_rm_vivax_train_pred = model_rm_vivax.predict(x_rm_vivax_train)
    y_rm_vivax_test_pred = model_rm_vivax.predict(x_rm_vivax_test)
    y_rm_vivax_blind_pred = model_rm_vivax.predict(x_rm_vivax_pred_blind)

    # Variables
    mae_rm_vivax = mean_absolute_error(y_rm_vivax_test, y_rm_vivax_test_pred)
    mse_rm_vivax = mean_squared_error(y_rm_vivax_test, y_rm_vivax_test_pred)

    print("=== Vivax ===")
    print("Mean Absolute Error (MAE): {0}".format(mae_rm_vivax))
    print("Mean Squared Error (MSE): {0}".format(mse_rm_vivax))

=== Falciparum ===
Mean Absolute Error (MAE): 0.028002445888749478
Mean Squared Error (MSE): 0.0022135962556478765

=== Vivax ===
Mean Absolute Error (MAE): 0.04925598029675747
Mean Squared Error (MSE): 0.006762645458121651


### Gráficas

In [62]:
# Falciparum

if x_rm_falciparum is not None:
    # Get the dates corresponding to each split

    train_rm_falciparum_dates = df_rm_sub.loc[x_rm_falciparum_train.index, 'fecha']
    test_rm_falciparum_dates = df_rm_sub.loc[x_rm_falciparum_test.index, 'fecha']
    blind_rm_falciparum_dates = df_rm_sub.loc[x_rm_falciparum_pred_blind.index, 'fecha']

    # Create DataFrames for each set with actual and predicted values
    df_train_rm_falciparum_plot = pd.DataFrame({
        'fecha': train_rm_falciparum_dates,
        'Actual': y_rm_falciparum_train,
        'Predicted': y_rm_falciparum_train_pred
    }).melt(id_vars=['fecha'], var_name='Type', value_name='IPA_falciparum_Value')
    df_train_rm_falciparum_plot['Set'] = 'Train'

    df_test_rm_falciparum_plot = pd.DataFrame({
        'fecha': test_rm_falciparum_dates,
        'Actual': y_rm_falciparum_test,
        'Predicted': y_rm_falciparum_test_pred
    }).melt(id_vars=['fecha'], var_name='Type', value_name='IPA_falciparum_Value')
    df_test_rm_falciparum_plot['Set'] = 'Test'

    df_blind_rm_falciparum_plot = pd.DataFrame({
        'fecha': blind_rm_falciparum_dates,
        'Actual': y_rm_falciparum_actual_blind,
        'Predicted': y_rm_falciparum_blind_pred
    }).melt(id_vars=['fecha'], var_name='Type', value_name='IPA_falciparum_Value')
    df_blind_rm_falciparum_plot['Set'] = 'Blind Prediction'

    # Concatenate all dataframes for plotting
    df_combined_rm_falciparum_plot = pd.concat([
        df_train_rm_falciparum_plot,
        df_test_rm_falciparum_plot,
        df_blind_rm_falciparum_plot,
    ])

    df_combined_rm_falciparum_plot = df_combined_rm_falciparum_plot \
                                        .groupby(["fecha", "Set", "Type"], as_index=False)["IPA_falciparum_Value"] \
                                            .mean()

    # Create the interactive time series plot
    fig_rm_falciparum = px.line(
        df_combined_rm_falciparum_plot,
        x='fecha',
        y='IPA_falciparum_Value',
        color='Set',
        line_dash='Type', # Use line style to distinguish Actual/Predicted
        title='IPA_falciparum Actual vs. Predicho - {0}'.format(rm_filters_title),
        labels={'fecha': 'Date', 'IPA_falciparum_Value': 'IPA Mensual'},
        hover_data={'fecha': '|%Y-%m-%d', 'IPA_falciparum_Value': ':.4f'}
    )

    fig_rm_falciparum.update_layout(
        hovermode='x unified', # Consolidate hover information
        xaxis_title='Fecha',
        yaxis_title='IPA Mensual'
    )
    fig_rm_falciparum.show()


# Vivax

if x_rm_vivax is not None:
    # Get the dates corresponding to each split

    train_rm_vivax_dates = df_rm_sub.loc[x_rm_vivax_train.index, 'fecha']
    test_rm_vivax_dates = df_rm_sub.loc[x_rm_vivax_test.index, 'fecha']
    blind_rm_vivax_dates = df_rm_sub.loc[x_rm_vivax_pred_blind.index, 'fecha']

    # Create DataFrames for each set with actual and predicted values
    df_train_rm_vivax_plot = pd.DataFrame({
        'fecha': train_rm_vivax_dates,
        'Actual': y_rm_vivax_train,
        'Predicted': y_rm_vivax_train_pred
    }).melt(id_vars=['fecha'], var_name='Type', value_name='IPA_vivax_Value')
    df_train_rm_vivax_plot['Set'] = 'Train'

    df_test_rm_vivax_plot = pd.DataFrame({
        'fecha': test_rm_vivax_dates,
        'Actual': y_rm_vivax_test,
        'Predicted': y_rm_vivax_test_pred
    }).melt(id_vars=['fecha'], var_name='Type', value_name='IPA_vivax_Value')
    df_test_rm_vivax_plot['Set'] = 'Test'

    df_blind_rm_vivax_plot = pd.DataFrame({
        'fecha': blind_rm_vivax_dates,
        'Actual': y_rm_vivax_actual_blind,
        'Predicted': y_rm_vivax_blind_pred
    }).melt(id_vars=['fecha'], var_name='Type', value_name='IPA_vivax_Value')
    df_blind_rm_vivax_plot['Set'] = 'Blind Prediction'

    # Concatenate all dataframes for plotting
    df_combined_rm_vivax_plot = pd.concat([
        df_train_rm_vivax_plot,
        df_test_rm_vivax_plot,
        df_blind_rm_vivax_plot,
    ])

    df_combined_rm_vivax_plot = df_combined_rm_vivax_plot \
                                    .groupby(["fecha", "Set", "Type"], as_index=False)["IPA_vivax_Value"] \
                                        .mean()

    # Create the interactive time series plot
    fig_rm_vivax = px.line(
        df_combined_rm_vivax_plot,
        x='fecha',
        y='IPA_vivax_Value',
        color='Set',
        line_dash='Type', # Use line style to distinguish Actual/Predicted
        title='IPA_vivax Actual vs. Predicho - {0}'.format(rm_filters_title),
        labels={'fecha': 'Date', 'IPA_vivax_Value': 'IPA Mensual'},
        hover_data={'fecha': '|%Y-%m-%d', 'IPA_vivax_Value': ':.4f'}
    )

    fig_rm_vivax.update_layout(
        hovermode='x unified', # Consolidate hover information
        xaxis_title='Fecha',
        yaxis_title='IPA Mensual'
    )
    fig_rm_vivax.show()

## Implementación y Calibración del Autómata Celular 2D

### Utilidades

In [69]:
class ACBoardType(Enum):
    HUMANS_FALCIPARUM_SUSCEPTIBLE = ("humans_falciparum_susceptible_count_board", "Humanos Susceptibles (Falciparum)", "YlGnBu")
    HUMANS_FALCIPARUM_INFECTED = ("humans_falciparum_infected_count_board", "Humanos Infectados (Falciparum)", "YlOrRd")
    HUMANS_FALCIPARUM_RECOVERED = ("humans_falciparum_recovered_count_board", "Humanos Recuperados (Falciparum)", "Greens")
    MOSQUITOES_FALCIPARUM_SUSCEPTIBLE = ("mosquitoes_falciparum_susceptible_count_board", "Mosquitos Susceptibles (Falciparum)", "PuBu")
    MOSQUITOES_FALCIPARUM_INFECTED = ("mosquitoes_falciparum_infected_count_board", "Mosquitos Infectados (Falciparum)", "OrRd")
    HUMANS_VIVAX_SUSCEPTIBLE = ("humans_vivax_susceptible_count_board", "Humanos Susceptibles (Vivax)", "YlGnBu")
    HUMANS_VIVAX_INFECTED = ("humans_vivax_infected_count_board", "Humanos Infectados (Vivax)", "YlOrRd")
    HUMANS_VIVAX_RECOVERED = ("humans_vivax_recovered_count_board", "Humanos Recuperados (Vivax)", "Greens")
    MOSQUITOES_VIVAX_SUSCEPTIBLE = ("mosquitoes_vivax_susceptible_count_board", "Mosquitos Susceptibles (Vivax)", "PuBu")
    MOSQUITOES_VIVAX_INFECTED = ("mosquitoes_vivax_infected_count_board", "Mosquitos Infectados (Vivax)", "OrRd")

    @property
    def key(self) -> str:
        return self.value[0]

    @property
    def label(self) -> str:
        return self.value[1]

    @property
    def cmap(self) -> str:
        return self.value[2]


def ac_get_array_module(prefer_gpu: bool = True) -> tuple[Any, bool]:
    """
    Retorna el módulo de arrays a utilizar (cupy si hay GPU CUDA disponible, de lo contrario numpy)
    y un booleano indicando si se está usando la GPU.

    Parameters:
        prefer_gpu (bool): Si se prefiere el uso de GPU con CUDA - No disponible en Mac OS

    Returns:
        tuple[Any, bool]: Librería a usar (NumPy o CUDA), Si está usando GPU o no
    """

    if prefer_gpu:
        try:
            import cupy as cp

            # Verificar que CUDA realmente responda
            device_count = cp.cuda.runtime.getDeviceCount()
            if device_count > 0:
                return cp, True
        except Exception:
            pass

    import numpy as np
    return np, False


### Variables

In [70]:
xp, is_using_cuda = ac_get_array_module()

print("Utilizando {0} v{1}".format("CUDA" if is_using_cuda else "NumPy", xp.__version__))

AC_CELL_POPULATION: int = 1_000
"""
Cantidad de habitantes representados por cada celda del tablero.
"""

AC_MAX_MOSQUITOES_PER_CELL: int = 2_000
"""
Capacidad de carga máxima de mosquitos vectores por celda (ratio máximo m = 2 mosquitos/humano).
"""

AC_BITING_RATE: float = 0.30
"""
Tasa diaria de picadura por mosquito (a).
"""

AC_MOSQUITO_MAX_AGE_DAYS: int = 21
"""
Esperanza de vida media del mosquito adulto en días (1 / mu_m).
"""

AC_MOSQUITO_WEATHER_DENSITY: float = 0.12
"""
Tasa de emergencia diaria de mosquitos modulada por el clima.
"""

AC_MOSQUITO_NEIGHBORS_DENSITY: float = 0.04
"""
Tasa de dispersión de mosquitos desde celdas vecinas.
"""

AC_NEIGHBOR_COUPLING: float = 0.25
"""
Fracción de exposición al contagio proveniente de las 8 celdas vecinas (vecindad de Moore).
"""

AC_STEPS_PER_MONTH: int = 4
"""
Sub-pasos de simulación por cada mes (4 = pasos semanales para acelerar la simulación vectorizada).
"""

# Parámetros específicos por cepa (Ross-Macdonald acoplado a clima)
AC_STRAIN_PARAMS: dict[MalariaType, dict[str, float]] = {
    MalariaType.FALCIPARUM: {
        "t_min": 18.0,                  # Temperatura mínima de esporogonia (°C)
        "t_opt": 28.0,                  # Temperatura óptima de transmisión (°C)
        "t_sigma": 4.5,                 # Amplitud térmica (°C)
        "prob_mosquito_to_human": 0.35, # Probabilidad de transmisión mosquito -> humano (b)
        "prob_human_to_mosquito": 0.30, # Probabilidad de transmisión humano -> mosquito (c)
        "recuperation_days": 14.0,      # Días promedio de infección humana (1 / gamma)
        "immunity_loss_days": 90.0,     # Días de inmunidad temporal antes de volver a S (1 / omega)
        "relapse_rate": 0.0,            # P. falciparum no presenta recaídas por hipnozoítos
    },
    MalariaType.VIVAX: {
        "t_min": 15.0,                  # P. vivax persiste en temperaturas más templadas (°C)
        "t_opt": 26.0,                  # Temperatura óptima de transmisión (°C)
        "t_sigma": 5.0,                 # Mayor tolerancia térmica (°C)
        "prob_mosquito_to_human": 0.30, # Probabilidad de transmisión mosquito -> humano (b)
        "prob_human_to_mosquito": 0.40, # Gametocitemia temprana facilita transmisión al vector (c)
        "recuperation_days": 21.0,      # Días promedio de infección humana (1 / gamma)
        "immunity_loss_days": 120.0,    # Días de inmunidad temporal (1 / omega)
        "relapse_rate": 0.0015,         # Tasa diaria de recaída por hipnozoítos hepáticos (R -> I)
    },
}

AC_SELECTED_MODEL_TYPE: ModelType = ModelType.COLOMBIA
AC_SELECTED_MALARIA_TYPE: MalariaType | list[MalariaType] = [MalariaType.VIVAX, MalariaType.FALCIPARUM]

# Set None for no filter
AC_SELECTED_REGION: Optional[type[Regions] | list[type[Regions]]]              = None
AC_SELECTED_DEPARTMENT: Optional[type[Departments] | list[type[Departments]]]  = None
AC_SELECTED_CITY: Optional[type[Cities] | list[type[Cities]]]                  = None

Utilizando NumPy v2.4.4


#### Utilidades para las variables

In [71]:
# Generate graphics filters title

ac_filters_title = f"{AC_SELECTED_MODEL_TYPE.value} - "

if AC_SELECTED_REGION is not None:
    if isinstance(AC_SELECTED_REGION, list):
        ac_filters_title += ", ".join([region.value for region in AC_SELECTED_REGION])
    else:
        ac_filters_title += AC_SELECTED_REGION.value
else:
    ac_filters_title += "Todas las regiones"

if AC_SELECTED_MODEL_TYPE in [ModelType.DEPARTMENT, ModelType.CITY]:
    ac_filters_title += " - "
    if AC_SELECTED_DEPARTMENT is not None:
        if isinstance(AC_SELECTED_DEPARTMENT, list):
            ac_filters_title += ", ".join([department.name.capitalize() for department in AC_SELECTED_DEPARTMENT])
        else:
            ac_filters_title += AC_SELECTED_DEPARTMENT.name.capitalize()
    else:
        ac_filters_title += "Todos los departamentos"

if AC_SELECTED_MODEL_TYPE == ModelType.CITY:
    ac_filters_title += " - "
    if AC_SELECTED_CITY is not None:
        if isinstance(AC_SELECTED_CITY, list):
            ac_filters_title += ", ".join([city.name.capitalize() for city in AC_SELECTED_CITY])
        else:
            ac_filters_title += AC_SELECTED_CITY.name.capitalize()
    else:
        ac_filters_title += "Todos los municipios"


# Get filtered DataFrame

df_ac_sub = {
    ModelType.COLOMBIA: df_colombia.copy(),
    ModelType.DEPARTMENT: df_departments.copy(),
    ModelType.CITY: df_municipal.copy(),
}[AC_SELECTED_MODEL_TYPE]

if AC_SELECTED_REGION is not None:
    if isinstance(AC_SELECTED_REGION, list):
        df_ac_sub = df_ac_sub[df_ac_sub["region"].isin([region.value for region in AC_SELECTED_REGION])]
    else:
        df_ac_sub = df_ac_sub[df_ac_sub["region"] == AC_SELECTED_REGION.value]

if AC_SELECTED_MODEL_TYPE in [ModelType.DEPARTMENT, ModelType.CITY] and AC_SELECTED_DEPARTMENT is not None:
    if isinstance(AC_SELECTED_DEPARTMENT, list):
        df_ac_sub = df_ac_sub[df_ac_sub["codigo_departamento"].isin([department.value for department in AC_SELECTED_DEPARTMENT])]
    else:
        df_ac_sub = df_ac_sub[df_ac_sub["codigo_departamento"] == AC_SELECTED_DEPARTMENT.value]

if AC_SELECTED_MODEL_TYPE == ModelType.CITY and AC_SELECTED_CITY is not None:
    if isinstance(AC_SELECTED_CITY, list):
        df_ac_sub = df_ac_sub[df_ac_sub["codigo_municipio"].isin([department.value for department in AC_SELECTED_CITY])]
    else:
        df_ac_sub = df_ac_sub[df_ac_sub["codigo_municipio"] == AC_SELECTED_CITY.value]


print(f"Título: {ac_filters_title}")
print(f"Tamaño del DataFrame: {len(df_ac_sub)} datos")

Título: Nivel Nacional - Todas las regiones
Tamaño del DataFrame: 1020 datos


### Utilidades para los tableros

In [ ]:
def ac_generate_initial_boards(
        xp: Any,
        df: Any,
):
    df_sorted = df.sort_values("fecha")
    df_by_date = df_sorted.groupby(["fecha"], sort=True)

    first_pop = float(df_by_date["poblacion_total"].sum().iloc[0])
    board_size = max(1, math.ceil(math.sqrt(first_pop / AC_CELL_POPULATION)))

    # Tomamos el promedio ponderado o media inicial de IPA (casos por cada 1.000 habitantes)
    IPA_falciparum = float(df_by_date["IPA_falciparum"].mean().iloc[0])
    IPA_vivax = float(df_by_date["IPA_vivax"].mean().iloc[0])

    p_init_falciparum = min(1.0, max(0.0, IPA_falciparum / float(AC_CELL_POPULATION)))
    p_init_vivax = min(1.0, max(0.0, IPA_vivax / float(AC_CELL_POPULATION)))

    # Humans - Falciparum (S + I + R = 1.000 por celda)
    humans_falciparum_infected_count_board = xp.random.binomial(
        n=AC_CELL_POPULATION,
        p=p_init_falciparum,
        size=(board_size, board_size),
    ).astype(xp.int32)
    humans_falciparum_recovered_count_board = xp.zeros((board_size, board_size), dtype=xp.int32)
    humans_falciparum_susceptible_count_board = (
        AC_CELL_POPULATION - humans_falciparum_infected_count_board
    ).astype(xp.int32)

    # Mosquitoes - Falciparum (Población basal inicial en equilibrio climático)
    init_mosq_total_f = xp.random.binomial(
        n=AC_MAX_MOSQUITOES_PER_CELL,
        p=0.25,
        size=(board_size, board_size),
    ).astype(xp.int32)
    mosquitoes_falciparum_infected_count_board = xp.random.binomial(
        n=init_mosq_total_f,
        p=min(1.0, p_init_falciparum * 2.0),
        size=(board_size, board_size),
    ).astype(xp.int32)
    mosquitoes_falciparum_susceptible_count_board = (
        init_mosq_total_f - mosquitoes_falciparum_infected_count_board
    ).astype(xp.int32)

    # Humans - Vivax (S + I + R = 1.000 por celda)
    humans_vivax_infected_count_board = xp.random.binomial(
        n=AC_CELL_POPULATION,
        p=p_init_vivax,
        size=(board_size, board_size),
    ).astype(xp.int32)
    humans_vivax_recovered_count_board = xp.zeros((board_size, board_size), dtype=xp.int32)
    humans_vivax_susceptible_count_board = (
        AC_CELL_POPULATION - humans_vivax_infected_count_board
    ).astype(xp.int32)

    # Mosquitoes - Vivax (Población basal inicial en equilibrio climático)
    init_mosq_total_v = xp.random.binomial(
        n=AC_MAX_MOSQUITOES_PER_CELL,
        p=0.25,
        size=(board_size, board_size),
    ).astype(xp.int32)
    mosquitoes_vivax_infected_count_board = xp.random.binomial(
        n=init_mosq_total_v,
        p=min(1.0, p_init_vivax * 2.0),
        size=(board_size, board_size),
    ).astype(xp.int32)
    mosquitoes_vivax_susceptible_count_board = (
        init_mosq_total_v - mosquitoes_vivax_infected_count_board
    ).astype(xp.int32)

    return {
        "board_size": board_size,
        "IPA_falciparum": IPA_falciparum,
        "IPA_vivax": IPA_vivax,
        "humans_falciparum_susceptible_count_board": humans_falciparum_susceptible_count_board,
        "humans_falciparum_infected_count_board": humans_falciparum_infected_count_board,
        "humans_falciparum_recovered_count_board": humans_falciparum_recovered_count_board,
        "mosquitoes_falciparum_susceptible_count_board": mosquitoes_falciparum_susceptible_count_board,
        "mosquitoes_falciparum_infected_count_board": mosquitoes_falciparum_infected_count_board,
        "humans_vivax_susceptible_count_board": humans_vivax_susceptible_count_board,
        "humans_vivax_infected_count_board": humans_vivax_infected_count_board,
        "humans_vivax_recovered_count_board": humans_vivax_recovered_count_board,
        "mosquitoes_vivax_susceptible_count_board": mosquitoes_vivax_susceptible_count_board,
        "mosquitoes_vivax_infected_count_board": mosquitoes_vivax_infected_count_board,
    }


def ac_count_neighbors(board: Any, xp: Any) -> Any:
    """
    Calcula la suma de los 8 vecinos de Moore en toda la cuadrícula 2D
    utilizando desplazamientos circulares (toroide) en GPU o CPU.

    Parameters:
        board: Tablero de conteos a analizar
        xp: Librería de NumPy o CuPy
    """

    int_board = board.astype(xp.int32)
    return (
        xp.roll(int_board,  1, axis=0) + xp.roll(int_board, -1, axis=0) +
        xp.roll(int_board,  1, axis=1) + xp.roll(int_board, -1, axis=1) +
        xp.roll(xp.roll(int_board,  1, axis=0),  1, axis=1) +
        xp.roll(xp.roll(int_board,  1, axis=0), -1, axis=1) +
        xp.roll(xp.roll(int_board, -1, axis=0),  1, axis=1) +
        xp.roll(xp.roll(int_board, -1, axis=0), -1, axis=1)
    )


def ac_to_numpy(arr: Any) -> np.ndarray:
    """Convierte un arreglo de CuPy o NumPy a np.ndarray en memoria de CPU."""
    if hasattr(arr, "get"):
        return arr.get()
    return np.asarray(arr)


def ac_fast_binomial(xp: Any, rng: Any, n: Any, p: Any) -> Any:
    """
    Muestreador Binomial vectorizado de alto rendimiento (aproximación Poisson para medias bajas
    y Gaussiana truncada en float32 para medias altas), evitando el cuello de botella de BTPE
    en tableros grandes (ej. Región Andina con 168x168 celdas).
    """
    mean = (n * p).astype(xp.float32)
    max_mean = float(xp.max(mean))
    if max_mean <= 0.0:
        return xp.zeros_like(n, dtype=xp.int32)
    if max_mean < 10.0:
        res = rng.poisson(xp.maximum(0.0, mean))
    else:
        std = xp.sqrt(xp.maximum(0.0, mean * (1.0 - p)))
        res = xp.rint(mean + std * rng.standard_normal(n.shape, dtype=xp.float32))
    return xp.clip(res, 0, n).astype(xp.int32)


def ac_step_strain_day(
        xp: Any,
        rng: Any,
        strain: MalariaType,
        S_h: Any,
        I_h: Any,
        R_h: Any,
        S_m: Any,
        I_m: Any,
        temperature: float,
        humidity: float,
        monthly_precipitation: float,
        dt_days: float = 1.0,
) -> tuple[Any, Any, Any, Any, Any, Any]:
    """
    Ejecuta un paso temporal (dt_days) del modelo metapoblacional SIRS (Humanos) - SI (Mosquitos)
    para una cepa específica (P. falciparum o P. vivax).
    Retorna los tableros actualizados (S_h, I_h, R_h, S_m, I_m) y los nuevos casos del sub-paso.
    """
    params = AC_STRAIN_PARAMS[strain]

    # 1. Factores climáticos de la cepa
    if params["t_min"] < temperature < 38.0:
        f_t = math.exp(-((temperature - params["t_opt"]) ** 2) / (2.0 * (params["t_sigma"] ** 2)))
    else:
        f_t = 0.0

    g_h = 1.0 / (1.0 + math.exp(-0.15 * (humidity - 65.0)))
    h_p = min(1.0, max(0.05, monthly_precipitation / 200.0))
    climate_suitability = f_t * g_h * h_p

    # 2. Conteo promedio de los 8 vecinos de Moore
    I_h_neighbors_mean = ac_count_neighbors(I_h, xp) / 8.0
    I_m_neighbors_mean = ac_count_neighbors(I_m, xp) / 8.0
    M_alive = S_m + I_m
    M_alive_neighbors_mean = ac_count_neighbors(M_alive, xp) / 8.0

    # 3. Exposición efectiva (acoplando la celda actual con sus 8 vecinos)
    I_m_effective = ((1.0 - AC_NEIGHBOR_COUPLING) * I_m + AC_NEIGHBOR_COUPLING * I_m_neighbors_mean) / float(AC_CELL_POPULATION)
    I_h_effective = ((1.0 - AC_NEIGHBOR_COUPLING) * I_h + AC_NEIGHBOR_COUPLING * I_h_neighbors_mean) / float(AC_CELL_POPULATION)

    # 4. Dinámica Humana (S -> I -> R -> S + recaídas en P. vivax) escalada por dt_days
    rate_m_to_h = max(0.0, min(1.0, AC_BITING_RATE * params["prob_mosquito_to_human"]))
    p_infect_human = xp.clip(1.0 - xp.power(1.0 - rate_m_to_h, I_m_effective * dt_days), 0.0, 1.0)
    p_recover_h = min(1.0, dt_days / params["recuperation_days"])
    p_loss_imm_h = min(1.0, dt_days / params["immunity_loss_days"])

    new_infected_h = ac_fast_binomial(xp, rng, S_h, p_infect_human)
    new_recovered_h = ac_fast_binomial(xp, rng, I_h, p_recover_h)

    if params["relapse_rate"] > 0.0:
        p_relapse = min(1.0, 1.0 - (1.0 - params["relapse_rate"]) ** dt_days)
        relapsed_h = ac_fast_binomial(xp, rng, R_h, p_relapse)
    else:
        relapsed_h = xp.zeros_like(R_h)

    remaining_R_h = xp.maximum(0, R_h - relapsed_h)
    lost_immunity_h = ac_fast_binomial(xp, rng, remaining_R_h, p_loss_imm_h)

    S_h_next = S_h - new_infected_h + lost_immunity_h
    I_h_next = I_h + new_infected_h - new_recovered_h + relapsed_h
    R_h_next = remaining_R_h + new_recovered_h - lost_immunity_h

    # 5. Dinámica del Mosquito (Nacimiento -> S_m -> I_m -> Muerte) escalada por dt_days
    p_death_m = min(1.0, dt_days / float(AC_MOSQUITO_MAX_AGE_DAYS))
    dead_S_m = ac_fast_binomial(xp, rng, S_m, p_death_m)
    dead_I_m = ac_fast_binomial(xp, rng, I_m, p_death_m)

    S_m_surv = S_m - dead_S_m
    I_m_surv = I_m - dead_I_m

    empty_m_slots = xp.maximum(0, AC_MAX_MOSQUITOES_PER_CELL - (S_m_surv + I_m_surv))
    p_emergence = AC_MOSQUITO_WEATHER_DENSITY * climate_suitability * dt_days
    p_dispersion = AC_MOSQUITO_NEIGHBORS_DENSITY * (M_alive_neighbors_mean / float(AC_MAX_MOSQUITOES_PER_CELL)) * dt_days
    p_birth_m = xp.clip(p_emergence + p_dispersion, 0.0, 1.0)

    newborn_m = ac_fast_binomial(xp, rng, empty_m_slots, p_birth_m)

    rate_h_to_m = max(0.0, min(1.0, AC_BITING_RATE * params["prob_human_to_mosquito"] * f_t))
    p_infect_mosquito = xp.clip(1.0 - xp.power(1.0 - rate_h_to_m, I_h_effective * dt_days), 0.0, 1.0)
    new_infected_m = ac_fast_binomial(xp, rng, S_m_surv, p_infect_mosquito)

    S_m_next = S_m_surv - new_infected_m + newborn_m
    I_m_next = I_m_surv + new_infected_m

    step_new_cases = new_infected_h + relapsed_h
    return S_h_next, I_h_next, R_h_next, S_m_next, I_m_next, step_new_cases


def ac_apply_evolution_rules(
        xp: Any,
        df: Any,
        ac_boards_data: Any,
) -> pd.DataFrame:
    """
    Evoluciona los tableros de P. falciparum y P. vivax a lo largo de las fechas del DataFrame,
    guardando tanto el historial numérico como las instantáneas 2D (frames_snapshots) para la animación.
    """
    rng = xp.random.default_rng()
    df_monthly = (
        df.groupby("fecha", as_index=False)
        .agg({
            "temperatura_media_Celsius": "mean",
            "humedad_mediana_pct": "mean",
            "precipitacion_media_mm": "mean",
            "IPA_falciparum": "mean",
            "IPA_vivax": "mean",
            "falciparum": "sum",
            "vivax": "sum",
        })
        .sort_values("fecha")
        .reset_index(drop=True)
    )

    total_cells = ac_boards_data["board_size"] ** 2

    # Estado actual Falciparum
    S_hf = ac_boards_data["humans_falciparum_susceptible_count_board"].copy()
    I_hf = ac_boards_data["humans_falciparum_infected_count_board"].copy()
    R_hf = ac_boards_data["humans_falciparum_recovered_count_board"].copy()
    S_mf = ac_boards_data["mosquitoes_falciparum_susceptible_count_board"].copy()
    I_mf = ac_boards_data["mosquitoes_falciparum_infected_count_board"].copy()

    # Estado actual Vivax
    S_hv = ac_boards_data["humans_vivax_susceptible_count_board"].copy()
    I_hv = ac_boards_data["humans_vivax_infected_count_board"].copy()
    R_hv = ac_boards_data["humans_vivax_recovered_count_board"].copy()
    S_mv = ac_boards_data["mosquitoes_vivax_susceptible_count_board"].copy()
    I_mv = ac_boards_data["mosquitoes_vivax_infected_count_board"].copy()

    history: list[dict[str, Any]] = []
    frames_snapshots: list[dict[str, Any]] = []

    for row in df_monthly.itertuples():
        days_in_month = float(pd.Timestamp(row.fecha).days_in_month)
        dt_days = days_in_month / float(AC_STEPS_PER_MONTH)
        temp = float(row.temperatura_media_Celsius)
        hum = float(row.humedad_mediana_pct)
        precip = float(row.precipitacion_media_mm)

        monthly_new_cases_f = 0
        monthly_new_cases_v = 0

        for _ in range(AC_STEPS_PER_MONTH):
            S_hf, I_hf, R_hf, S_mf, I_mf, new_f = ac_step_strain_day(
                xp, rng, MalariaType.FALCIPARUM, S_hf, I_hf, R_hf, S_mf, I_mf, temp, hum, precip, dt_days
            )
            S_hv, I_hv, R_hv, S_mv, I_mv, new_v = ac_step_strain_day(
                xp, rng, MalariaType.VIVAX, S_hv, I_hv, R_hv, S_mv, I_mv, temp, hum, precip, dt_days
            )
            monthly_new_cases_f += int(xp.sum(new_f))
            monthly_new_cases_v += int(xp.sum(new_v))

        frames_snapshots.append({
            "fecha": pd.Timestamp(row.fecha),
            "humans_falciparum_susceptible_count_board": ac_to_numpy(S_hf).copy(),
            "humans_falciparum_infected_count_board": ac_to_numpy(I_hf).copy(),
            "humans_falciparum_recovered_count_board": ac_to_numpy(R_hf).copy(),
            "mosquitoes_falciparum_susceptible_count_board": ac_to_numpy(S_mf).copy(),
            "mosquitoes_falciparum_infected_count_board": ac_to_numpy(I_mf).copy(),
            "humans_vivax_susceptible_count_board": ac_to_numpy(S_hv).copy(),
            "humans_vivax_infected_count_board": ac_to_numpy(I_hv).copy(),
            "humans_vivax_recovered_count_board": ac_to_numpy(R_hv).copy(),
            "mosquitoes_vivax_susceptible_count_board": ac_to_numpy(S_mv).copy(),
            "mosquitoes_vivax_infected_count_board": ac_to_numpy(I_mv).copy(),
        })

        history.append({
            "fecha": row.fecha,
            "falciparum_S_h": int(xp.sum(S_hf)),
            "falciparum_I_h": int(xp.sum(I_hf)),
            "falciparum_R_h": int(xp.sum(R_hf)),
            "falciparum_S_m": int(xp.sum(S_mf)),
            "falciparum_I_m": int(xp.sum(I_mf)),
            "falciparum_nuevos_casos_mes": monthly_new_cases_f,
            "IPA_falciparum_simulado": monthly_new_cases_f / float(total_cells),
            "IPA_falciparum_real": float(row.IPA_falciparum),
            "vivax_S_h": int(xp.sum(S_hv)),
            "vivax_I_h": int(xp.sum(I_hv)),
            "vivax_R_h": int(xp.sum(R_hv)),
            "vivax_S_m": int(xp.sum(S_mv)),
            "vivax_I_m": int(xp.sum(I_mv)),
            "vivax_nuevos_casos_mes": monthly_new_cases_v,
            "IPA_vivax_simulado": monthly_new_cases_v / float(total_cells),
            "IPA_vivax_real": float(row.IPA_vivax),
        })

    ac_boards_data["humans_falciparum_susceptible_count_board"] = S_hf
    ac_boards_data["humans_falciparum_infected_count_board"] = I_hf
    ac_boards_data["humans_falciparum_recovered_count_board"] = R_hf
    ac_boards_data["mosquitoes_falciparum_susceptible_count_board"] = S_mf
    ac_boards_data["mosquitoes_falciparum_infected_count_board"] = I_mf

    ac_boards_data["humans_vivax_susceptible_count_board"] = S_hv
    ac_boards_data["humans_vivax_infected_count_board"] = I_hv
    ac_boards_data["humans_vivax_recovered_count_board"] = R_hv
    ac_boards_data["mosquitoes_vivax_susceptible_count_board"] = S_mv
    ac_boards_data["mosquitoes_vivax_infected_count_board"] = I_mv

    df_sim = pd.DataFrame(history)
    ac_boards_data["simulation_history"] = df_sim
    ac_boards_data["frames_snapshots"] = frames_snapshots
    return df_sim


def ac_run_evolution_parallel(xp: Any, data_store: dict[str, Any]) -> None:
    """
    Ejecuta la evolución del modelo SIR en paralelo para todas las regiones/departamentos/municipios
    almacenados en data_store utilizando ThreadPoolExecutor.
    """
    start = time.perf_counter()
    max_workers = min(len(data_store), os.cpu_count() or 4)

    def _worker(item: tuple[str, Any]) -> tuple[str, pd.DataFrame]:
        entity_name, entity_data = item
        df_res = ac_apply_evolution_rules(xp, entity_data["df"], entity_data)
        return entity_name, df_res

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        list(executor.map(_worker, data_store.items()))

    elapsed = time.perf_counter() - start
    if elapsed >= 60:
        print(f"Tiempo de evolución paralela ({len(data_store)} entidades): {(elapsed / 60):.2f} m")
    else:
        print(f"Tiempo de evolución paralela ({len(data_store)} entidades): {elapsed:.2f} s")


#### Utilidades para la animación

In [ ]:
def _resolve_entity_keys(
        data_store: dict[str, Any],
        entity_filter: Optional[Any],
) -> list[str]:
    """Resuelve el filtro de región/departamento/municipio a las llaves presentes en ac_data_store."""
    if entity_filter is None:
        return list(data_store.keys())

    items = entity_filter if isinstance(entity_filter, list) else [entity_filter]
    resolved: list[str] = []
    for item in items:
        if isinstance(item, str) and item in data_store:
            resolved.append(item)
        elif isinstance(item, Enum):
            # Para Regions el valor es el nombre; para Departments/Cities el nombre del Enum coincide con la llave
            if str(item.value) in data_store:
                resolved.append(str(item.value))
            else:
                for k in data_store.keys():
                    if k.upper() == item.name.upper():
                        resolved.append(k)
                        break

    if not resolved:
        raise ValueError(f"No se encontró '{entity_filter}' dentro de ac_data_store: {list(data_store.keys())}")

    return resolved


def ac_animate_boards(
        data_store: dict[str, Any],
        entity_filter: Optional[Any] = None,
        boards_filter: Optional[ACBoardType | list[ACBoardType]] = None,
        frame_step: int = 1,
        interval_ms: int = 120,
) -> HTML:
    """
    Genera la animación HTML interactiva (estilo Proyecto.ipynb) de la evolución espacial
    de los tableros del modelo SIR.

    - Si entity_filter apunta a una sola entidad y boards_filter es None, muestra los 10 tableros
      (2 filas x 5 columnas: Falciparum arriba, Vivax abajo).
    - Si entity_filter es None (todas las entidades), organiza una cuadrícula donde cada fila es
      una entidad y cada columna un tablero seleccionado (o una cuadrícula 2D de entidades si se
      elige un único tablero).
    """
    entity_keys = _resolve_entity_keys(data_store, entity_filter)

    if boards_filter is None:
        if len(entity_keys) == 1:
            selected_boards = list(ACBoardType)
        else:
            # Cuando se visualizan todas las entidades a la vez, por defecto mostramos los 4 tableros de infectados
            selected_boards = [
                ACBoardType.HUMANS_FALCIPARUM_INFECTED,
                ACBoardType.MOSQUITOES_FALCIPARUM_INFECTED,
                ACBoardType.HUMANS_VIVAX_INFECTED,
                ACBoardType.MOSQUITOES_VIVAX_INFECTED,
            ]
    elif isinstance(boards_filter, ACBoardType):
        selected_boards = [boards_filter]
    else:
        selected_boards = list(boards_filter)

    first_snapshots = data_store[entity_keys[0]].get("frames_snapshots")
    if not first_snapshots:
        raise RuntimeError("Primero debes ejecutar ac_run_evolution_parallel(xp, ac_data_store).")

    frame_indices = list(range(0, len(first_snapshots), max(1, frame_step)))

    # Determinar disposición de la cuadrícula (filas x columnas)
    if len(entity_keys) == 1 and len(selected_boards) == 10:
        nrows, ncols = 2, 5
        layout_mode = "single_entity_10_boards"
    elif len(entity_keys) == 1:
        ncols = min(5, len(selected_boards))
        nrows = math.ceil(len(selected_boards) / ncols)
        layout_mode = "single_entity"
    elif len(selected_boards) == 1:
        ncols = min(4, math.ceil(math.sqrt(len(entity_keys))))
        nrows = math.ceil(len(entity_keys) / ncols)
        layout_mode = "multi_entity_single_board"
    else:
        nrows = len(entity_keys)
        ncols = len(selected_boards)
        layout_mode = "multi_entity_grid"

    fig, axes = plt.subplots(
        nrows=nrows,
        ncols=ncols,
        figsize=(max(4.0, 2.8 * ncols), max(3.5, 2.7 * nrows)),
        dpi=72,
        squeeze=False,
    )

    init_date_str = first_snapshots[0]["fecha"].strftime("%b %Y")
    title_prefix = entity_keys[0] if len(entity_keys) == 1 else ac_filters_title
    suptitle_obj = fig.suptitle(f"{title_prefix} — {init_date_str}", fontsize=12, fontweight="bold")

    # Precalcular vmax por cada par (entidad, tablero) para mantener la escala de color estable
    plot_items: list[tuple[Any, str, str]] = []

    for ax in axes.flat:
        ax.set_xticks([])
        ax.set_yticks([])

    if layout_mode in ("single_entity_10_boards", "single_entity"):
        ent = entity_keys[0]
        snaps = data_store[ent]["frames_snapshots"]
        for idx, b_type in enumerate(selected_boards):
            r, c = divmod(idx, ncols)
            ax = axes[r, c]
            ax.set_title(b_type.label, fontsize=8.5)
            vmax = max(1, int(max(np.max(s[b_type.key]) for s in snaps)))
            vmin = int(min(np.min(s[b_type.key]) for s in snaps)) if "susceptible" in b_type.key else 0
            im = ax.imshow(snaps[0][b_type.key], cmap=b_type.cmap, vmin=vmin, vmax=vmax)
            fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
            plot_items.append((im, ent, b_type.key))
        for idx in range(len(selected_boards), nrows * ncols):
            r, c = divmod(idx, ncols)
            axes[r, c].axis("off")
    elif layout_mode == "multi_entity_single_board":
        b_type = selected_boards[0]
        for idx, ent in enumerate(entity_keys):
            r, c = divmod(idx, ncols)
            ax = axes[r, c]
            snaps = data_store[ent]["frames_snapshots"]
            ax.set_title(f"{ent}\n{b_type.label}", fontsize=9)
            vmax = max(1, int(max(np.max(s[b_type.key]) for s in snaps)))
            vmin = int(min(np.min(s[b_type.key]) for s in snaps)) if "susceptible" in b_type.key else 0
            im = ax.imshow(snaps[0][b_type.key], cmap=b_type.cmap, vmin=vmin, vmax=vmax)
            fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
            plot_items.append((im, ent, b_type.key))
        for idx in range(len(entity_keys), nrows * ncols):
            r, c = divmod(idx, ncols)
            axes[r, c].axis("off")
    else:
        for r, ent in enumerate(entity_keys):
            snaps = data_store[ent]["frames_snapshots"]
            for c, b_type in enumerate(selected_boards):
                ax = axes[r, c]
                if r == 0:
                    ax.set_title(b_type.label, fontsize=8.5)
                if c == 0:
                    ax.set_ylabel(ent, fontsize=9, fontweight="bold")
                vmax = max(1, int(max(np.max(s[b_type.key]) for s in snaps)))
                vmin = int(min(np.min(s[b_type.key]) for s in snaps)) if "susceptible" in b_type.key else 0
                im = ax.imshow(snaps[0][b_type.key], cmap=b_type.cmap, vmin=vmin, vmax=vmax)
                fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
                plot_items.append((im, ent, b_type.key))

    fig.tight_layout(rect=[0, 0, 1, 0.95])

    def update_frame(frame_idx: int):
        current_date = first_snapshots[frame_idx]["fecha"].strftime("%b %Y")
        suptitle_obj.set_text(f"{title_prefix} — {current_date}")
        for im, ent, b_key in plot_items:
            im.set_data(data_store[ent]["frames_snapshots"][frame_idx][b_key])
        return [im for im, _, _ in plot_items]

    anim = animation.FuncAnimation(
        fig,
        update_frame,
        frames=frame_indices,
        interval=interval_ms,
        blit=False,
    )

    plt.close(fig)

    start = time.perf_counter()
    result_html = HTML(anim.to_jshtml())
    end = time.perf_counter()

    time_diff_in_seconds = end - start
    if time_diff_in_seconds >= 60:
        print(f"Tiempo de renderizado de animación: {(time_diff_in_seconds / 60):.2f} m")
    else:
        print(f"Tiempo de renderizado de animación: {time_diff_in_seconds:.2f} s")

    return result_html


### Tableros iniciales

In [73]:
ac_data_store = {}
 
ac_data = df_ac_sub.groupby({
    ModelType.COLOMBIA: "region",
    ModelType.DEPARTMENT: "departamento",
    ModelType.CITY: "municipio",
}[AC_SELECTED_MODEL_TYPE])

for region, new_df in ac_data:
    ac_data_store[region] = {
        "df": new_df,
        **ac_generate_initial_boards(xp, new_df),
    }

print(ac_data_store.keys())

dict_keys(['Amazonia', 'Andina', 'Caribe', 'Orinoquia', 'Pacifico'])


### Aplicación de evolución

In [119]:
ac_run_evolution_parallel(xp, ac_data_store)

Tiempo de evolución paralela (5 entidades): 10.76 s


In [120]:
t_data_filter = "Amazonia"

t_data = ac_data_store[t_data_filter]
t_data["simulation_history"].head(10)

,fecha,falciparum_S_h,falciparum_I_h,falciparum_R_h,falciparum_S_m,falciparum_I_m,falciparum_nuevos_casos_mes,IPA_falciparum_simulado,IPA_falciparum_real,vivax_S_h,vivax_I_h,vivax_R_h,vivax_S_m,vivax_I_m,vivax_nuevos_casos_mes,IPA_vivax_simulado,IPA_vivax_real
0,2007-01-01,311639,77289,511072,1647190,152810,172033,191.147778,0.0396,168095,123606,608299,1477475,322525,182960,203.288889,0.1409
1,2007-02-01,319360,74207,506433,1270084,135826,151010,167.788889,0.1002,169316,122538,608146,1170002,292474,164354,182.615556,0.2783
2,2007-03-01,343525,66918,489557,1678149,121851,148484,164.982222,0.1328,178323,120090,601587,1499340,300660,173755,193.061111,0.3668
3,2007-04-01,357012,69343,473645,1666453,121456,147477,163.863333,0.2911,175122,123972,600906,1484715,315285,178472,198.302222,0.4530
4,2007-05-01,363592,70547,465861,1678603,119437,155703,173.003333,0.2352,172799,124553,602648,1486026,313974,184349,204.832222,0.4577
5,2007-06-01,372593,66640,460767,1608803,105783,145624,161.804444,0.1223,173276,123061,603663,1492882,307118,175630,195.144444,0.3622
6,2007-07-01,386906,65031,448063,1663129,103353,143343,159.270000,0.1246,174881,121809,603310,1495386,304614,180051,200.056667,0.4577
7,2007-08-01,391078,67736,441186,1670607,108477,148997,165.552222,0.1211,174760,122829,602411,1493711,306289,181459,201.621111,0.4425
8,2007-09-01,383936,73282,442782,1658679,120278,154420,171.577778,0.1339,172459,124046,603495,1483294,316706,177586,197.317778,0.4495
9,2007-10-01,369760,75560,454680,1672955,127045,167676,186.306667,0.0978,171841,123732,604427,1486722,313278,183307,203.674444,0.3564


### Gráficas

#### Vivax

In [107]:
humans_vivax_susceptibles_fig = px.line(
    data_frame=t_data["simulation_history"],
    x=t_data["simulation_history"]["fecha"],
    y=t_data["simulation_history"]["vivax_S_h"],
    labels={
        "vivax_S_h": "Humanos susceptibles"
    },
    template="plotly_dark",
)

humans_vivax_infected_fig = px.line(
    data_frame=t_data["simulation_history"],
    x=t_data["simulation_history"]["fecha"],
    y=t_data["simulation_history"]["vivax_I_h"],
    labels={
        "vivax_I_h": "Humanos infectados",
    },
    template="plotly_dark",
)

mosquitoes_vivax_susceptibles_fig = px.line(
    data_frame=t_data["simulation_history"],
    x=t_data["simulation_history"]["fecha"],
    y=t_data["simulation_history"]["vivax_S_m"],
    labels={
        "vivax_S_m": "Mosquitos susceptibles"
    },
    template="plotly_dark",
)

mosquitoes_vivax_infected_fig = px.line(
    data_frame=t_data["simulation_history"],
    x=t_data["simulation_history"]["fecha"],
    y=t_data["simulation_history"]["vivax_I_m"],
    labels={
        "vivax_I_m": "Mosquitos infectados",
    },
    template="plotly_dark",
)

In [121]:
go.Figure(
    data=humans_vivax_susceptibles_fig.data + humans_vivax_infected_fig.data,
    layout={
        "title": f"Humanos susceptibles vs infectados (Vivax) - {t_data_filter}"
    },
)

In [122]:
go.Figure(
    data=mosquitoes_vivax_susceptibles_fig.data + mosquitoes_vivax_infected_fig.data,
    layout={
        "title": f"Mosquitos susceptibles vs infectados (Vivax) - {t_data_filter}"
    },
)

In [123]:
go.Figure(
    data=mosquitoes_vivax_infected_fig.data + humans_vivax_infected_fig.data,
    layout={
        "title": f"Infección Mosquitos vs Humanos (Vivax) - {t_data_filter}"
    },
)

#### Falciparum

In [114]:
humans_falciparum_susceptibles_fig = px.line(
    data_frame=t_data["simulation_history"],
    x=t_data["simulation_history"]["fecha"],
    y=t_data["simulation_history"]["falciparum_S_h"],
    labels={
        "falciparum_S_h": "Humanos susceptibles"
    },
    template="plotly_dark",
)

humans_falciparum_infected_fig = px.line(
    data_frame=t_data["simulation_history"],
    x=t_data["simulation_history"]["fecha"],
    y=t_data["simulation_history"]["falciparum_I_h"],
    labels={
        "falciparum_I_h": "Humanos infectados",
    },
    template="plotly_dark",
)

mosquitoes_falciparum_susceptibles_fig = px.line(
    data_frame=t_data["simulation_history"],
    x=t_data["simulation_history"]["fecha"],
    y=t_data["simulation_history"]["falciparum_S_m"],
    labels={
        "falciparum_S_m": "Mosquitos susceptibles"
    },
    template="plotly_dark",
)

mosquitoes_falciparum_infected_fig = px.line(
    data_frame=t_data["simulation_history"],
    x=t_data["simulation_history"]["fecha"],
    y=t_data["simulation_history"]["falciparum_I_m"],
    labels={
        "falciparum_I_m": "Mosquitos infectados",
    },
    template="plotly_dark",
)

In [124]:
go.Figure(
    data=humans_falciparum_susceptibles_fig.data + humans_falciparum_infected_fig.data,
    layout={
        "title": f"Humanos susceptibles vs infectados (Falciparum) - {t_data_filter}"
    },
)

In [125]:
go.Figure(
    data=mosquitoes_falciparum_susceptibles_fig.data + mosquitoes_falciparum_infected_fig.data,
    layout={
        "title": f"Mosquitos susceptibles vs infectados (Falciparum) - {t_data_filter}"
    },
)

In [126]:
go.Figure(
    data=mosquitoes_falciparum_infected_fig.data + humans_falciparum_infected_fig.data,
    layout={
        "title": f"Infección Mosquitos vs Humanos (Falciparum) - {t_data_filter}"
    },
)

#### Filtro y animación de la evolución de los tableros

In [79]:
AC_ANIMATION_ENTITY_FILTER: Optional[Any] = Regions.ANDINA
"""
Filtro de visualización:
- `None`: Muestra TODAS las regiones/departamentos/municipios.
- `Regions.`: Muestra una región en específico (Solo para el modelo Nacional)
- `Departments.`: Muestra un departamento en específico (Solo para el modelo Departamental)
- `Cities.`: Muestra un municipio en específico (Solo para el modelo Municipal)
"""

AC_ANIMATION_BOARDS_FILTER: Optional[ACBoardType | list[ACBoardType]] = None
"""
Tipos de tablero a mostrar:
- `None`: Muestra todos los tableros de (Falciparum y Vivax) si el filtro de visualización está para una sola región/departamento/municipio.
- `None`: Muestra los tableros de infectados (Humanos y Mosquitos) para Falciparum y Vivax cuando el filtro de visualización es general.
- Se puede seleccionar uno o varios tableros en específico del Enum para visualizarlos por separado.
"""

AC_ANIMATION_FRAME_STEP: int = 1
"""
Paso a paso de cada frame (En meses), es decir, si el valor está en 1 pasará de mes en mes, si está en 2
pasará cada 2 meses la visualización de la animación.
"""

ac_animate_boards(
    data_store=ac_data_store,
    entity_filter=AC_ANIMATION_ENTITY_FILTER,
    boards_filter=AC_ANIMATION_BOARDS_FILTER,
    frame_step=AC_ANIMATION_FRAME_STEP,
    interval_ms=120,
)

Tiempo de renderizado de animación: 35.90 s


## Visualización Científica y Tablero Analítico

## Discusión de Resultados y Limitaciones del Modelo

1. Los modelos tuvieron que ser entrenados de diferente forma para el dataset Nacional, Departamental y/o Municipal, entonces la labor investigativa para encontrar las variables acordes para cada modelo fue algo compleja porque miden cosas diferentes.

2. El modelo SIR es demasiado extenso para el caso de querer mostrarlo por cada cepa/región/departamento/municipio, hace muy complicado ver la animación de manera correcta, pero el experimento me parece muy bueno para aplicar los conocimientos vistos en clase y de otra materia como HPC.

3. Al inicio, empecé a analizar todos los casos por iguales, pero me di cuenta que el dataset tenía información por cepa de Malaria y decidí adentrarme en elaborar un modelo para cada cepa después de investigar sobre cada porque se desarrollan en diferentes condiciones climáticas.

4. Es interesante ver cómo una cepa es más dominante que otra en algunas regiones/departamentos/municipios, esto en base al clima de cada región.

5. La cepa que predomina en Colombia es la Vivax, pero la falciparum tiene algunos ciclos de contagios en algunos años específicos.

## Declaración de uso de IA

Para el desarrollo, estructuración matemática y optimización computacional de este proyecto se utilizó asistencia de Inteligencia Artificial Generativa bajo la modalidad de programación asistida (*pair programming*), verificando y validando manualmente cada fórmula epidemiológica, regla de transición y visualización generada.

### Modelo y versión utilizado

- **Modelo:** Gemini 4 Argon (Medium) — Google DeepMind.
- **Rol del sistema:** Agente de codificación avanzada y asistencia técnica en ciencia de datos y modelado matemático.

### Entorno utilizado

- **Plataforma / Interfaz de IA:** Google Antigravity IDE (Agente integrado en el entorno de desarrollo con acceso al contexto del espacio de trabajo y notebooks de Jupyter).
- **Sistema Operativo y Hardware:** macOS ejecutándose en un `Apple MacBook Pro` con procesador **Apple M1 Pro** (10 núcleos físicos/lógicos: 8 de alto rendimiento *Firestorm* + 2 de alta eficiencia *Icestorm*) y **16 GB de memoria unificada LPDDR5**.
- **Entorno de ejecución de software:**
  - **Python:** `3.14.3`
  - **Pandas:** `3.0.2`
  - **NumPy:** `2.4.4`
  - **Matplotlib:** `3.10.8`
  - **Scikit-Learn:** `1.9.1`

### Alcance del uso en el proyecto

1. **Formulación matemática del Autómata Celular SIR/SIRS acoplado (Ross-Macdonald):** Definición de las ecuaciones de transición estocástica por cepa (*Plasmodium falciparum* y *Plasmodium vivax*) en metapoblaciones de $N_H = 1\,000$ humanos por celda, incluyendo la inicialización binomial a partir del Índice Parasitario Anual (IPA) y la ventana de coexistencia de coinfección.
2. **Optimización y vectorización computacional:** Implementación de muestreo estocástico aproximado vectorizado en `float32` (Poisson para tasas bajas y aproximación Normal truncada para tasas altas) y ejecución concurrente por región/departamento/municipio mediante `ThreadPoolExecutor`.
3. **Visualización científica e interactividad:** Adaptación del motor de animación HTML/JS (`matplotlib.animation.FuncAnimation`) con filtros independientes por entidad geográfica y por tipo de tablero epidemiológico.

### Registro de prompts representativos (Entrada exacta y síntesis de salida)

| # | Prompt de entrada (Texto exacto) | Síntesis de la salida generada |
| :-: | :--- | :--- |
| **1** | *"Necesito implementar el modelo SIR para malaria falciparum y malaria vivax, ya tengo los datasets, pero necesito saber las fórmulas para cada una de las cepas para aplicar las reglas de evolución del modelo. Cada celda del modelo equivale a 1.000 humanos y el total de celdas es la cantidad de habitantes de la región/departamento/municipio. [...] Con esos tableros se puede hacer el modelo o los tengo que agrupar de manera diferente? [...] Inicialmente hice que el conteo de infectados se base en el IPA y lo haga para cada celda, pero es un análisis correcto o se debe hacer con probabilidad estadística y que sea menor o igual al valor resultante del IPA de la cepa?"* | Validación de los 12 tableros de estado acoplados (humanos y mosquitos por cepa); justificación estadística de por qué reemplazar la asignación determinista del IPA por un muestreo estocástico $\text{Binomial}(N_H=1000, p_s = \text{clip}(\text{IPA}_s/1000, 0, 1))$; y formulación matemática de las reglas de evolución Ross-Macdonald diferenciando los parámetros biológicos de *P. falciparum* y *P. vivax* (tasas de picadura, infectividad, recuperación y recaída por hipnozoítos). |
| **2** | *"puedes aplicarlo en el modelo sir del código?"* | Implementación en el notebook del diccionario de parámetros biológicos `AC_STRAIN_PARAMS`, refactorización de `ac_generate_initial_boards` con inicialización estocástica binomial por celda y programación de `ac_count_neighbors`, `ac_step_strain_day` y `ac_apply_evolution_rules` integrando factores climáticos (temperatura, humedad y precipitación), vecindad de Moore ($3 \times 3$) y límite de coexistencia por coinfección. |
| **3** | *"Puedes poner la animación del notebook de Proyecto.ipynb de la sección 'Aplicación del modelo SIR' en el final_project.ipynb en la sección de 'Aplicación de evolución' adaptado a los diferentes tableros que se pueden ver de cada región... Si es posible con un filtro a parte donde se pueda elegir si ver un solo tablero de una región/departamento/municipio específico o si ver todos en una cuadrícula. Paraleliza las ejecuciones si es posible para que no demore tanto en cargar la visualización."* | Creación del enumerador `ACBoardType`, paralelización multihilo de la evolución de todas las entidades geográficas mediante `ac_run_evolution_parallel` (`ThreadPoolExecutor`), aceleración vectorizada del muestreo binomial (`ac_fast_binomial`) reduciendo el tiempo de cómputo nacional a ~10.8 s, e implementación de `ac_animate_boards` con filtros independientes (`AC_ANIMATION_ENTITY_FILTER` y `AC_ANIMATION_BOARDS_FILTER`) para renderizar vistas individuales o cuadrículas comparativas en HTML/JS. |
| **4** | *"Puedes añadirme en la declaración de uso de IA lo siguiente: - Modelo y versión utilizado - Entorno utilizado - Añadir una tabla con 5 prompts al azar con el texto de entrada exacto y el de salida (Si son muy extensos no agregues este punto)"* | Estructuración de la sección `## Declaración de uso de IA` detallando el modelo empleado (`Gemini 4 Argon (Medium)`), el entorno de desarrollo (`Google Antigravity IDE` sobre macOS M1 Pro con Python `3.14.3`) y el alcance metodológico de la asistencia de IA en el proyecto. |
| **5** | *"sí, añade una tabla con el síntesis de salida para no llenar de info el notebook"* | Incorporación de la presente tabla resumen en el notebook, conservando el texto íntegro de las consultas realizadas junto con una síntesis ejecutiva de las salidas técnicas y matemáticas obtenidas. |